<a href="https://colab.research.google.com/github/covinolab/sbi_tutorial_algosb2026/blob/main/02_sbi_double_well.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SBI for time series: Brownian motion in a double well

**AlgoSB 2026 · simulation-based inference · hands-on notebook 2 of 2 · 75 minutes**

Notebook 1 had an answer key. This one does not. A particle diffuses in a double-well potential, you see its position over time, and you want two numbers: the **height of the barrier** between the wells and the **diffusion coefficient**. This is the model behind the force-spectroscopy example of this morning's lecture, without the linker: you observe the particle position directly.

The simulator is given. Your work is to decide **what to measure on a trajectory**, and to find out what those measurements can and cannot tell you.

| Part | Min | Your challenge |
| --- | --- | --- |
| 0 | 5 | Recap and setup |
| 1 | 10 | Inference by hand |
| 2 | 33 | Invent your summary statistics |
| 3 | 12 | Catch the blind trajectory |
| 4 | 15 | A posterior that is blind to D |

The rules are those of Notebook 1: every part is a challenge, a check cell prints ✅ PASS or ❌ NOT YET, how you get there is up to you, and after a pass you write one sentence on why it worked. Worked answers are in `solutions/`.

In [ ]:
# Setup. On Colab this installs sbi and fetches the tutorial files; on your own machine it does nothing.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("tutorial_utils.py"):
    !pip install -q sbi==0.27.0
    !git clone -q --depth 1 https://github.com/covinolab/sbi_tutorial_algosb2026.git
    %cd sbi_tutorial_algosb2026

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

import sbi
from sbi.analysis import pairplot
from sbi.inference import NPE
from sbi.utils import BoxUniform

import tutorial_utils as tu
from tutorial_utils import (check_blind, check_blind_to_D, check_practice, check_score,
                            plot_posteriors, plot_trajectories, posterior_widths, summarize,
                            train_and_score)

warnings.filterwarnings("ignore")  # sbi is talkative; none of its warnings matter today
tu.set_plot_style()
torch.manual_seed(0)
np.random.seed(0)
print(f"sbi {sbi.__version__} · torch {torch.__version__}")

## Part 0 · Recap and setup (5 min)

The five steps are the same as in Notebook 1, with the same names. Only the data got harder.

| Step | In the code | Notebook 1 | This notebook |
| --- | --- | --- | --- |
| 1. Prior | `prior` | box for a 2D mean | box for barrier height and diffusion coefficient |
| 2. Simulator | `simulator(theta)` returns `x` | one noisy 2D point | one trajectory of 5,000 positions |
| 3. Training set | `theta`, `x` | 2D points | trajectories, reduced to **summary statistics** |
| 4. Training | `posterior` | NPE | NPE |
| 5. Inference and checks | `x_o`, `samples` | compare with the answer key | a blind trajectory, and no answer key |

**The model.** A particle at position $x$ feels the double-well potential

$$G_0(x) = \Delta G^{\ddagger}\, f(x/x^{\ddagger}), \qquad f(u) = \begin{cases} 1 - 2u^2, & |u| \le 1/2 \\ 2(|u|-1)^2, & |u| > 1/2 \end{cases}$$

with wells at $x = \pm x^{\ddagger}$ and a barrier of height $\Delta G^{\ddagger}$ at $x = 0$. It moves by overdamped Brownian dynamics with diffusion coefficient $D$:

$$x(t+\Delta t) = x(t) - D\,\partial_x G_0(x)\,\Delta t + \sqrt{2 D\,\Delta t}\;\xi(t), \qquad \xi \sim \mathcal{N}(0, 1).$$

Units: $k_BT = 1$ and $x^{\ddagger} = 1$. The two parameters to infer are $\theta = (\Delta G^{\ddagger},\ \log_{10} D)$.

In [ ]:
# 1. Prior: barrier height between 2 and 6 k_BT, diffusion coefficient between 0.1 and 1
prior = BoxUniform(low=torch.tensor([2.0, -1.0]), high=torch.tensor([6.0, 0.0]))

LIMITS = [[2, 6], [-1, 0]]
LABELS = tu.DW_LABELS

# The potential for three barrier heights
grid = np.linspace(-1.8, 1.8, 400)
fig, ax = plt.subplots(figsize=(5.5, 3.2))
for dG, color in zip((2, 4, 6), (tu.C_YOURS, tu.C_KEY, tu.C_OTHER)):
    ax.plot(grid, tu.double_well_potential(grid, dG), lw=2, color=color, label=rf"$\Delta G^\ddagger$ = {dG} $k_BT$")
ax.set(xlabel="position $x$", ylabel=r"$G_0(x)$ [$k_BT$]", ylim=(-0.3, 8))
ax.legend();

## Part 1 · Inference by hand (10 min)

Here is the simulator. Read it together: it is the Langevin equation above, integrated with a time step of 0.002. Every 50th step is saved, which gives 5,000 frames, 0.1 time units apart.

In [ ]:
DT, STRIDE, N_FRAMES = 0.002, 50, 5000

def force(x, dG):
    """Force -dG_0/dx: a harmonic barrier for |x| <= 1/2, matched to harmonic wells at x = ±1."""
    return -dG * np.where(np.abs(x) <= 0.5, -4 * x, 4 * (x - np.sign(x)))

# 2. Simulator
def simulator(theta, seed=None):
    """Brownian dynamics in the double well, one trajectory per row of theta.

    theta: shape (n, 2), columns barrier height [k_BT] and log10 D
    returns x: positions, shape (n, 5000)
    """
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    dG, D = theta[:, 0], 10 ** theta[:, 1]
    rng = np.random.default_rng(seed)
    drift = D * DT                                        # D Δt, multiplies the force
    kick = np.sqrt(2 * D * DT)                            # size of one random kick
    x_t = np.full(len(theta), -1.0, dtype=np.float32)     # every trajectory starts in the left well
    x = np.empty((len(theta), N_FRAMES), dtype=np.float32)
    for frame in range(N_FRAMES):
        xi = rng.standard_normal((STRIDE, len(theta)), dtype=np.float32)
        for step in range(STRIDE):
            x_t = x_t + drift * force(x_t, dG) + kick * xi[step]
        x[:, frame] = x_t
    return torch.from_numpy(x)

In [ ]:
# Two parameter sets, two trajectories. Each plot shows the whole trajectory, its first frames, and its histogram.
x_demo = simulator(torch.tensor([[2.5, np.log10(0.8)], [5.0, np.log10(0.2)]]), seed=1)
plot_trajectories(x_demo, [r"$\Delta G^\ddagger$ = 2.5, D = 0.8", r"$\Delta G^\ddagger$ = 5.0, D = 0.2"]);

Now the inverse problem. Somebody simulated the **practice trajectory** below and did not tell you the parameters.

**Challenge.** Find its $\Delta G^{\ddagger}$ and $D$ by trial and error: simulate, compare, adjust.

**Passes when** your last try is within 0.5 $k_BT$ of the hidden barrier height and within a factor of 1.5 of the hidden diffusion coefficient, in at most ten tries.

In [ ]:
x_practice = tu.load_trajectory("practice")

tries = []        # every try is recorded here; run this cell again to start over

def try_parameters(dG, D):
    """Simulate one trajectory at (dG, D) and show it next to the practice trajectory."""
    tries.append((dG, D))
    x_try = simulator(torch.tensor([[dG, np.log10(D)]]), seed=1000 + len(tries))[0]
    plot_trajectories([x_practice, x_try],
                      ["practice trajectory", rf"try {len(tries)}: $\Delta G^\ddagger$ = {dG}, D = {D}"])
    return x_try

In [ ]:
x_try = try_parameters(dG=5.0, D=0.2)        # ← change the numbers and run again

In [ ]:
check_practice(tries)

**✍️ One sentence.** What did you look at to get the barrier height, and what to get $D$? Could the number of hops alone have given you both?

## Part 2 · Invent your summary statistics (33 min)

What you just did by eye, a network should now do: map a trajectory to the parameters behind it. But a trajectory is 5,000 numbers, and you have time for 2,000 simulations. So each trajectory is first reduced to a handful of numbers, its **summary statistics**, and the network is trained on those. The network cannot use information the summaries threw away.

First, the training set. This is step 3 of the recipe, simulated live:

In [ ]:
# 3. Training set: 2,000 parameters from the prior, one trajectory each (about 10 seconds)
torch.manual_seed(0)
theta = prior.sample((2000,))
x = simulator(theta, seed=0)
print("theta:", tuple(theta.shape), "  x:", tuple(x.shape))

A summary function takes a batch of trajectories and returns one row of numbers per trajectory. Here is the simplest one you could think of, as a baseline: the mean and the standard deviation of $x$.

In [ ]:
def summary_baseline(x):
    """x: NumPy array of shape (n_trajectories, 5000). Returns shape (n_trajectories, n_statistics)."""
    return np.stack([
        x.mean(axis=1),
        x.std(axis=1),
    ], axis=1)

print(summarize(summary_baseline, x[:3]))

The helper `train_and_score` runs steps 3 and 4 exactly as in Notebook 1 and adds a score. Its core is:

```python
s = summarize(summary, x)                           # your statistics, for all 2,000 trajectories
inference = NPE(prior=prior, density_estimator=...) # a small neural spline flow
inference.append_simulations(theta, s).train()
posterior = inference.build_posterior()
score = posterior.log_prob(theta_test, x=s_test).mean()
```

**The score** is the average log posterior density at the true parameters, over a common test set of 200 trajectories that nobody trains on. Higher is better. It rewards posteriors that are narrow and in the right place, and it punishes overconfident ones. A posterior that learned nothing and returns the prior scores −1.39.

`posterior_widths` adds a second view: how wide the 90% interval of each parameter is, as a fraction of the prior's, on average over the test set.

In [ ]:
posterior_baseline, score_baseline = train_and_score(summary_baseline, theta, x)
posterior_widths(posterior_baseline, summary_baseline)
check_score(score_baseline)

The baseline barely beats the prior.

**Challenge.** Reduce the 5,000 frames to a handful of numbers that carry what a trajectory says about $\Delta G^{\ddagger}$ and $D$. Train, read the score, improve. There is no list of statistics to choose from and no hint: you know what you looked at in Part 1.

**Passes when**
- 🥉 bronze: you beat the baseline,
- 🥈 silver: you reach the score of our reference solution,
- 🥇 gold: you beat it.

Retraining the same statistics with another `seed` moves the score by about ±0.05, so differences smaller than that mean nothing.

In [ ]:
def my_summary(x):
    """x: NumPy array of shape (n_trajectories, 5000). Returns shape (n_trajectories, n_statistics)."""
    return np.stack([
        x.mean(axis=1),
        x.std(axis=1),
        # ← your statistics
    ], axis=1)

posterior, score = train_and_score(my_summary, theta, x)
posterior_widths(posterior, my_summary)
check_score(score)

**🗣 On the board:** the best score in the room.

**Three questions for when you have a medal.**

1. Which of your statistics moved the score most, and why?
2. Does adding statistics always help? Add ten columns of pure noise to your summary and look.
3. Two groups reach the same score with different statistics. What does that say about the statistics?

## Part 3 · Catch the blind trajectory (12 min)

This is the trajectory the afternoon is about. Its parameters are hidden; they are revealed at the end of the notebook.

In [ ]:
x_o = tu.load_trajectory("blind")
plot_trajectories([x_o], ["blind trajectory"]);

**Challenge.** Infer the parameters of the blind trajectory with your best statistics. The check cell knows the truth and does not show it.

**Passes when** the 90% region of your posterior contains the truth **and** covers less than 5% of the prior area.

The cell below is step 5 of the recipe. The observation has to be reduced with the same summary function the network was trained on. If the check fails, go back to Part 2 and improve your statistics. (No statistics yet? `samples = tu.load_fallback_posterior("reference").sample((10_000,), x=x_o)` uses our pre-trained reference posterior.)

In [ ]:
# 5. Inference: reduce the observation with your statistics, then sample the posterior
samples = posterior.sample((10_000,), x=summarize(my_summary, x_o))

pairplot(samples, limits=LIMITS, labels=LABELS, figsize=(5, 5));

In [ ]:
check_blind(samples)

**✍️ One sentence.** Which is worse: a posterior that is too wide, or one that is narrow and misses the truth? Which of the two does the check punish?

## Part 4 · A posterior that is blind to D (15 min)

Now use the score helper as an instrument. Statistics decide what the posterior can know. So you can also decide what it cannot.

**Challenge.** Choose statistics so that the posterior learns $\Delta G^{\ddagger}$ and nothing about $D$.

**Passes when**, on the test set, the 90% interval of $\Delta G^{\ddagger}$ is narrower than 25% of the prior's, while the 90% interval of $\log_{10} D$ stays wider than 85% of the prior's.

**Then try the reverse:** statistics that give $D$ and nothing about $\Delta G^{\ddagger}$. There is no check for this one. How far do you get, and what stops you?

In [ ]:
def summary_blind_to_D(x):
    return np.stack([
        x.mean(axis=1),
        x.std(axis=1),
        # ← replace these by your statistics
    ], axis=1)

posterior_blind_to_D, _ = train_and_score(summary_blind_to_D, theta, x)
check_blind_to_D(posterior_blind_to_D, summary_blind_to_D)

**✍️ One sentence.** What do all statistics that are blind to $D$ have in common?

## Wrap-up

The hidden parameters of the blind trajectory, as promised:

In [ ]:
truth = tu.reveal_blind_truth()
plot_posteriors(samples, "your posterior", truth=truth, truth_label="hidden truth",
                limits=LIMITS, axis_labels=LABELS);

In [ ]:
tu.scorecard()

**What you did.** The same five steps as in Notebook 1, on a time series: prior, simulator, training set, training, inference. The new step was yours: deciding what to measure. You judged the result by a score on held-out simulations, by whether the posterior contains the truth, and by how much it narrows the prior.

**Next steps for real data.**

- **A measurement model.** In a force-spectroscopy experiment you do not see the molecule. You see a bead attached to it by a flexible linker, and the bead has its own diffusion coefficient. Put the linker and the instrument noise into the simulator, and the same workflow infers the molecular parameters through them. This is the model of the lecture and of the [reference notebook](https://github.com/covinolab/sbi_for_smfe_tutorial).
- **Learned summaries.** Instead of inventing statistics, let an embedding network learn them together with the posterior (bonus 4 below). On raw data that needs many more simulations and much longer training, and it can find statistics nobody thought of.
- **Sequential rounds.** If you care about one observation only, simulate where its posterior is: train, sample parameters from the posterior of your observation, simulate there, retrain. Fewer simulations, at the price of a network that is good for this one observation only.
- **Checks without an answer key.** Coverage (Notebook 1, Part 4) and posterior predictive checks (bonus 1 below) work on any simulator.

---
## Bonus challenges

For those who finish early.

### Bonus 1 · Spot the wrong simulator

Three more blind trajectories. Two come from the simulator of this notebook. One comes from a **tilted** double well, a potential your simulator cannot produce. An SBI posterior will answer for all three without complaint.

Find the impostor with a **posterior predictive check**: for each trajectory, sample parameters from its posterior, simulate new trajectories with them, and compare those with the trajectory itself. If the model is right, the observation looks like one of its own predictions.

**Passes when** the check cell confirms your pick.

In [ ]:
x_bonus = tu.load_trajectory("bonus")        # shape (3, 5000)
for i in range(3):
    plot_trajectories([x_bonus[i]], [f"trajectory {i}"])

In [ ]:
# Your posterior predictive check



# tu.check_wrong_simulator(pick)       # pick = 0, 1 or 2

### Bonus 2 · Sparse training data

Simulations are often expensive. Cut the training set from 2,000 simulations to 200, `theta[:200]` and `x[:200]`, and watch the score drop. Then win it back with those 200 simulations: change the statistics, the network (`density_estimator=`) or the training (any argument of sbi's `.train()`, for example `training_batch_size=`).

**Passes when**, with 200 simulations, you are back above bronze and have won back at least half of what the cut cost you.

In [ ]:
# Your training with 200 simulations



# tu.check_sparse(score_200, score_200_start=..., score_2000=...)

### Bonus 3 · An ensemble of posteriors: what does it buy?

Train five networks on the same 200 simulations, each with a different `seed`, and combine them with `EnsemblePosterior` from `sbi.inference`. Three steps:

1. **Look.** Plot the posterior of the blind trajectory for two single networks and for the ensemble.
2. **Measure the gain.** Score the ensemble on the test set and subtract the mean score of its five members. `tu.log_probs_at_truth(posterior, my_summary)` returns the 200 numbers whose mean is the score, and its docstring says how to combine them for an ensemble. Then do the same with all 2,000 simulations.
3. **Check the intervals.** `tu.interval_coverage(posterior_or_list, my_summary)` tells you how often the 50% interval contains the truth on the test set, for one network and for the list of five.

**Passes when** the ensemble gains more than 0.2 with 200 simulations, and at least twice what it gains with 2,000.

Five trainings on 200 simulations take about ten seconds, five on 2,000 about a minute.

In [ ]:
# Your ensemble



# tu.check_ensemble(gain_200=..., gain_2000=...)

### Bonus 4 · Learned summaries: give the network the right raw material

Let a network invent the statistics. An *embedding network* sits in front of the density estimator, `posterior_nn(model="nsf", embedding_net=...)`, and the two are trained together.

The obvious input is the raw trajectory, 5,000 numbers. Here that works badly and is slow (the table in the solutions has the numbers): the network has to find out by itself what in a trajectory matters. So help it with one fact you know. **The trajectory is a Markov chain:** the next frame depends on the present one only. Everything the trajectory says about the parameters is then in how often it steps from one position to another.

1. **Count the transitions.** Write `transition_histogram(x)`: for each trajectory the 2D histogram of (position now, position one frame later), 24 by 24 bins between −2 and 2, divided by the number of steps. For one trajectory `row`, `np.histogram2d(row[:-1], row[1:], bins=24, range=[[-2, 2], [-2, 2]])[0]` does the counting. Return one row of 576 numbers per trajectory.
2. **Let a network read it.** Put `FCEmbedding(input_dim=576, output_dim=8, num_layers=2, num_hiddens=64)` from `sbi.neural_nets.embedding_nets` in front of the flow, and train with `train_and_score(transition_histogram, theta, x, density_estimator=...)`. A histogram is one object, not 576 unrelated inputs: with `z_score_x="structured"` in `posterior_nn`, sbi standardizes it with a single mean and a single standard deviation.
3. **Take the dynamics away.** Do the same with the histogram of positions alone (`np.histogram`, 24 numbers) and look at `posterior_widths`.

**Passes when** the learned summary scores above 2.5.

Each training takes about ten seconds. This remains a bonus because learned summaries are in general much more expensive than hand-made statistics: an embedding network that reads the raw trajectory needs many more simulations, and its training takes minutes to hours instead of seconds. The histogram is what makes a learned summary fit into this afternoon.

In [ ]:
# Your learned summary



# tu.check_learned_summary(score_learned)